# pyMOFL: Quickstart and Common Workflows

Welcome to **pyMOFL** (Python Modular Optimization Function Library).
This walkthrough demonstrates:
1. Direct function instantiation and evaluation
2. Vector and scalar transformations via `ComposedFunction`
3. Config-driven instantiation with `FunctionFactory`
4. Function surface and contour visualization with `matplotlib`


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from pyMOFL.factories import FunctionFactory
from pyMOFL.functions.benchmark import RastriginFunction, SphereFunction
from pyMOFL.functions.transformations import (
    BiasTransform,
    ComposedFunction,
    RotateTransform,
    ShiftTransform,
)


## 1. Direct Benchmark Functions
Benchmark functions support both single-vector `evaluate(x)` and batch `evaluate_batch(X)` evaluations.


In [ ]:
# Initialize a 2D Sphere function
sphere = SphereFunction(dimension=2)
x = np.array([1.0, 2.0])
print("Sphere(1, 2) =", sphere.evaluate(x))

# Vectorized batch evaluation
X = np.array([
    [0.0, 0.0],
    [1.0, 1.0],
    [2.0, 2.0],
])
print("Batch evaluation =", sphere.evaluate_batch(X))

# Check global minimum
x_opt, f_opt = sphere.get_global_minimum()
print(f"Global minimum: f({x_opt}) = {f_opt}")


## 2. Transformations and Composition Pipeline
Functions can be shifted, rotated, scaled, and biased using `ComposedFunction`.


In [ ]:
base = RastriginFunction(dimension=2)
shift = np.array([2.0, -1.0])
# 45-degree rotation matrix
theta = np.pi / 4
rot = np.array([
    [np.cos(theta), -np.sin(theta)],
    [np.sin(theta), np.cos(theta)]
])

composed = ComposedFunction(
    base_function=base,
    input_transforms=[
        ShiftTransform(shift),
        RotateTransform(rot),
    ],
    output_transforms=[BiasTransform(100.0)]
)

print("Composed evaluation at shift point:", composed.evaluate(shift))


## 3. Config-Driven Instantiation
`FunctionFactory` builds complex functions from declarative JSON/dictionary configurations.


In [ ]:
factory = FunctionFactory()

config = {
    "type": "bias",
    "parameters": {"value": 50.0},
    "function": {
        "type": "shift",
        "parameters": {"vector": [1.5, -2.5]},
        "function": {
            "type": "ackley",
            "parameters": {"dimension": 2}
        }
    }
}

func = factory.create_function(config)
print("Factory created function evaluation at [1.5, -2.5]:", func.evaluate(np.array([1.5, -2.5])))


## 4. Visualizing Landscapes
Plotting the 2D contour and 3D surface of a benchmark function.


In [ ]:
# Create grid
x = np.linspace(-5.12, 5.12, 100)
y = np.linspace(-5.12, 5.12, 100)
X, Y = np.meshgrid(x, y)
grid_points = np.column_stack([X.ravel(), Y.ravel()])

rastrigin = RastriginFunction(dimension=2)
Z = rastrigin.evaluate_batch(grid_points).reshape(X.shape)

# Plot 2D Contour and 3D Surface
fig = plt.figure(figsize=(12, 5))

ax1 = fig.add_subplot(1, 2, 1)
cs = ax1.contourf(X, Y, Z, levels=50, cmap="viridis")
fig.colorbar(cs, ax=ax1)
ax1.set_title("2D Contour: Rastrigin Function")
ax1.set_xlabel("x1")
ax1.set_ylabel("x2")

ax2 = fig.add_subplot(1, 2, 2, projection="3d")
surf = ax2.plot_surface(X, Y, Z, cmap="plasma", edgecolor="none", alpha=0.9)
ax2.set_title("3D Surface: Rastrigin Function")
ax2.set_xlabel("x1")
ax2.set_ylabel("x2")
ax2.set_zlabel("f(x1, x2)")

plt.tight_layout()
plt.show()
